# Model experiments

Design choices tested **outside the Airflow DAG**, on the existing datamart. The chosen approach is implemented in `scripts/model_train.py`, which the DAG runs.

Same split as the pipeline (lab 5 structure, model train date 2024-05-01):

| Set | Labels observed | Loans started |
|---|---|---|
| Train / test (random 80 / 20) | 2023-07 to 2024-02 | 2023-01 to 2023-08 |
| Validation | 3-fold CV inside train | |
| OOT | 2024-03 to 2024-04 | 2023-09 to 2023-10 |

1. Does clickstream add value?
2. How should Logistic Regression handle nulls (and is winsorizing needed)?
3. How important are the columns with nulls?
4. Model selection: best validation Gini vs least overfitting

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyspark
import xgboost as xgb
from pyspark.sql.functions import col
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# same helpers as the pipeline
from scripts.model_train import CAT_COLS, NON_FEATURE_COLS, gini, smallest_gap_index

spark = pyspark.sql.SparkSession.builder.appName("dev").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/24 17:23:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## data: same split as the pipeline

In [2]:
labels = spark.read.parquet("datamart/gold/label_store/gold_label_store_*.parquet") \
    .filter(col("snapshot_date").between("2023-07-01", "2024-04-30")) \
    .select("loan_id", "label", col("snapshot_date").alias("label_date"))
data = labels.join(spark.read.parquet("datamart/gold/feature_store/gold_feature_store_*.parquet"), on="loan_id").toPandas()

all_num = [c for c in data.columns if c not in NON_FEATURE_COLS + CAT_COLS + ["label_date"]]
click_cols = [c for c in all_num if c.startswith("fe_")]
base_cols = [c for c in all_num if c not in click_cols]

oot = data[data["label_date"] >= pd.Timestamp("2024-03-01").date()]
train_test = data[data["label_date"] < pd.Timestamp("2024-03-01").date()]
X_train, X_test, y_train, y_test = train_test_split(train_test[all_num + CAT_COLS], train_test["label"],
                                                    test_size=0.2, random_state=88, stratify=train_test["label"])
X_oot, y_oot = oot[all_num + CAT_COLS], oot["label"]
null_cols = [c for c in all_num if X_train[c].isna().any()]
print("train", len(X_train), "| test", len(X_test), "| OOT", len(X_oot))

train 3229 | test 808 | OOT 941


## helpers

In [3]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=88)

class Winsorizer(BaseEstimator, TransformerMixin):
    # caps each column at its 1st / 99th percentile, learned on the training rows only
    def fit(self, X, y=None):
        self.lower_, self.upper_ = np.nanpercentile(X, 1, axis=0), np.nanpercentile(X, 99, axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)

def logistic_regression(num_cols, nulls="median_flag"):
    steps = {"median_flag": [("impute", SimpleImputer(strategy="median", add_indicator=True))],
             "median": [("impute", SimpleImputer(strategy="median"))],
             "winsor_median_flag": [("winsor", Winsorizer()), ("impute", SimpleImputer(strategy="median", add_indicator=True))],
             "none": []}[nulls]
    pipe = Pipeline([
        ("prep", ColumnTransformer([("num", Pipeline(steps + [("scale", StandardScaler())]), num_cols),
                                    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)])),
        ("model", LogisticRegression(solver="liblinear", max_iter=1000, random_state=88)),
    ])
    return GridSearchCV(pipe, {"model__C": [0.001, 0.01, 0.1, 1, 10], "model__penalty": ["l1", "l2"],
                               "model__class_weight": [None, "balanced"]},
                        scoring="roc_auc", cv=cv, return_train_score=True, refit=smallest_gap_index, n_jobs=-1)

def xgboost(num_cols, refit=smallest_gap_index):
    pipe = Pipeline([
        ("prep", ColumnTransformer([("num", "passthrough", num_cols),
                                    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)])),
        ("model", xgb.XGBClassifier(eval_metric="logloss", tree_method="hist", random_state=88)),
    ])
    return RandomizedSearchCV(pipe, {
        "model__n_estimators": [100, 200, 400, 800], "model__max_depth": [1, 2, 3, 4],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1], "model__min_child_weight": [5, 10, 20, 40],
        "model__subsample": [0.6, 0.8, 1.0], "model__colsample_bytree": [0.5, 0.7, 1.0],
        "model__gamma": [0, 0.5, 1, 2], "model__reg_lambda": [1, 5, 10, 20], "model__reg_alpha": [0, 1, 5],
    }, n_iter=50, scoring="roc_auc", cv=cv, return_train_score=True, refit=refit, random_state=42, n_jobs=-1)

def evaluate(name, search, num_cols):
    cols = num_cols + CAT_COLS
    search.fit(X_train[cols], y_train)
    model = search.best_estimator_
    return {"experiment": name,
            "train": gini(y_train, model.predict_proba(X_train[cols])[:, 1]),
            "validation": 2 * search.cv_results_["mean_test_score"][search.best_index_] - 1,
            "test": gini(y_test, model.predict_proba(X_test[cols])[:, 1]),
            "OOT": gini(y_oot, model.predict_proba(X_oot[cols])[:, 1])}, model

def table(rows):
    return pd.DataFrame(rows).set_index("experiment").round(4)

## 1. does clickstream add value?

All train, test and OOT loans here have clickstream. New applicants from 2024-07 onwards do not, which is why the model goes live in 2024-05.

In [4]:
rows, models = [], {}
for model_name, make in [("LR", logistic_regression), ("XGB", xgboost)]:
    for click_name, num_cols in [("no clickstream", base_cols), ("+ clickstream", all_num)]:
        row, models[(model_name, click_name)] = evaluate(f"{model_name} | {click_name}", make(num_cols), num_cols)
        rows.append(row)
table(rows)

,train,validation,test,OOT
experiment,,,,
LR | no clickstream,0.5532,0.5237,0.5527,0.5549
LR | + clickstream,0.6739,0.6439,0.6669,0.7180
XGB | no clickstream,0.6461,0.5850,0.6021,0.6016
XGB | + clickstream,0.7269,0.6700,0.6905,0.7308


## 2. Logistic Regression: how to handle nulls

Silver turns invalid values into nulls. Logistic Regression cannot take nulls; XGBoost handles them natively. Winsorizing here is done properly (caps learned on the training rows only).

In [5]:
kept_cols = [c for c in all_num if c not in null_cols]
table([
    evaluate("median + missing flag (pipeline)", logistic_regression(all_num, "median_flag"), all_num)[0],
    evaluate("median only", logistic_regression(all_num, "median"), all_num)[0],
    evaluate("winsorize + median + missing flag", logistic_regression(all_num, "winsor_median_flag"), all_num)[0],
    evaluate("drop columns with nulls", logistic_regression(kept_cols, "none"), kept_cols)[0],
])

,train,validation,test,OOT
experiment,,,,
median + missing flag (pipeline),0.6739,0.6439,0.6669,0.7180
median only,0.6709,0.6470,0.6716,0.7198
winsorize + median + missing flag,0.6786,0.6479,0.6722,0.7204
drop columns with nulls,0.6374,0.6158,0.6492,0.6916


## 3. how important are the columns with nulls?

In [6]:
print("null rate in train (%):")
print((X_train[null_cols].isna().mean() * 100).round(1).sort_values(ascending=False).to_string())

cols = all_num + CAT_COLS
importance = {}
for model_name in ["LR", "XGB"]:
    imp = permutation_importance(models[(model_name, "+ clickstream")], X_test[cols], y_test, scoring="roc_auc",
                                 n_repeats=10, random_state=88, n_jobs=-1)
    importance[model_name] = pd.Series(imp.importances_mean, index=cols)
    share = importance[model_name][null_cols].clip(lower=0).sum() / importance[model_name].clip(lower=0).sum()
    print(f"{model_name}: columns with nulls hold {share:.0%} of total importance")

pd.DataFrame({f"{m} importance": importance[m][null_cols] for m in importance}).sort_values("XGB importance", ascending=False).round(4)

null rate in train (%):
Amount_invested_monthly       4.5
Num_of_Loan_clean             4.1
Total_EMI_per_month_clean     3.5
Age_clean                     3.1
Num_Credit_Card_clean         2.6
delayed_payment_rate          2.3
Changed_Credit_Limit          2.1
Interest_Rate_clean           2.0
Num_Credit_Inquiries_clean    1.7
Num_Bank_Accounts_clean       1.4


LR: columns with nulls hold 30% of total importance


XGB: columns with nulls hold 9% of total importance


,LR importance,XGB importance
Changed_Credit_Limit,0.0098,0.0069
Interest_Rate_clean,0.0153,0.0031
Num_Credit_Inquiries_clean,0.0084,0.0015
Num_Credit_Card_clean,0.0095,0.0015
delayed_payment_rate,0.0014,0.0006
Amount_invested_monthly,0.0024,0.0003
Total_EMI_per_month_clean,0.0000,0.0003
Num_of_Loan_clean,0.0015,0.0000
Num_Bank_Accounts_clean,-0.0014,-0.0000
Age_clean,0.0001,-0.0001


## 4. model selection: best validation vs least overfitting

The pipeline's rule: among settings within 0.02 of the best validation Gini, pick the one with the smallest train-validation gap.

In [7]:
best_validation = lambda cv_results: int(np.argmax(cv_results["mean_test_score"]))
table([
    evaluate("XGB | best validation Gini", xgboost(all_num, refit=best_validation), all_num)[0],
    evaluate("XGB | smallest gap within 0.02 (pipeline)", xgboost(all_num), all_num)[0],
]).assign(train_val_gap=lambda t: (t["train"] - t["validation"]).round(4))

,train,validation,test,OOT,train_val_gap
experiment,,,,,
XGB | best validation Gini,0.9104,0.687,0.7002,0.7491,0.2234
XGB | smallest gap within 0.02 (pipeline),0.7269,0.670,0.6905,0.7308,0.0569


In [8]:
spark.stop()